# HadISD Data Download Notebook

This notebook will help you download a subset of the HadISD dataset directly from the Met Office website. The data will be stored in a user-specified directory (or a sensible default), and extracted for further processing (e.g., conversion to Zarr).

- **Source:** [HadISD v3.4.0.2023f](https://www.metoffice.gov.uk/hadobs/hadisd/v340_2023f/download.html)
- **Instructions:**
    1. Set the download directory (or use the default).
    2. Download the data using Python's `requests` package.
    3. Extract the `.tar.gz` archive.
    4. The extracted files will be ready for use in the next notebook (`HadISD_to_zarr.ipynb`).

> **Note:** Download size is large. Ensure you have sufficient disk space and a stable internet connection.

In [ ]:
import requests
from tqdm.auto import tqdm
import tarfile
import gzip
import shutil

### Retrieve Path to Download Directory
The download location will default to a folder named "HadISD_data" in your home directory.<br>
If you want to change this, you can do so in the `Data_config.ipynb` configuration notebook. <br>


In [ ]:
%run Data_Config.ipynb
print(f"Data will be downloaded to: {download_dir}")   

### Download HadISD Data
The following code will download the HadISD data files. Some files take longer to download than others depending on time of day. To download different WMO datasets, you can change `wmo_id_range` in the `Data_Config.ipynb` notebook .

The full list of available data can be found here:
https://www.metoffice.gov.uk/hadobs/hadisd/v340_2023f/download.html

In [ ]:
print(f"Downloading HadISD data for WMO range: {wmo_id_range}")

In [ ]:
wmo_id_range = wmo_id_range # This has been defined in HadISD_data_config.ipynb

wmo_str = f"WMO_{wmo_id_range}"
url = f"https://www.metoffice.gov.uk/hadobs/hadisd/v340_2023f/data/{wmo_str}.tar.gz"
tar_name = f"{wmo_str}.tar"
filename = download_dir / tar_name

print(f"Downloading HadISD data for WMO range {wmo_id_range} from {url}")

In [ ]:
# Download with resume support and progress bar
headers = {}
initial_pos = 0
if filename.exists():
    initial_pos = filename.stat().st_size
    headers['Range'] = f'bytes={initial_pos}-'
    mode = 'ab'
else:
    mode = 'wb'

response = requests.get(url, stream=True, headers=headers)
total = int(response.headers.get('content-length', 0)) + initial_pos

with open(filename, mode) as f, tqdm(
    desc=f"Downloading {filename.name}",
    total=total,
    initial=initial_pos,
    unit='B', unit_scale=True, unit_divisor=1024
) as bar:
    for chunk in response.iter_content(chunk_size=8192):
        if chunk:
            f.write(chunk)
            bar.update(len(chunk))

print(f"Download complete: {filename}")

### Extract Tar Files and Move to Netcdf Subfolder

In [ ]:
# Extract the tar.gz file
extract_dir = download_dir / tar_name.replace('.tar', '')       
extract_dir.mkdir(exist_ok=True)

with tarfile.open(filename, "r:gz") as tar:
    tar.extractall(path=extract_dir)

print(f"Extraction complete. Files are in: {extract_dir}")

In [ ]:
# Create subfolder for netcdf
netcdf_dir = download_dir / tar_name.replace('.tar', '')  / "netcdf"
netcdf_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# Move extracted .nc files into netcdf_dir after extraction
for gz_path in extract_dir.glob('*.nc.gz'):
    nc_path = gz_path.with_suffix('')  # Remove .gz extension
    with gzip.open(gz_path, 'rb') as f_in, open(nc_path, 'wb') as f_out:
        f_out.write(f_in.read())
    print(f"Extracted: {nc_path}")
    gz_path.unlink()  # Delete the .gz file after extraction
    print(f"Deleted: {gz_path}")
    # Move the .nc file to netcdf_dir
    shutil.move(str(nc_path), netcdf_dir / nc_path.name)
    print(f"Moved: {nc_path} -> {netcdf_dir / nc_path.name}")

print("All .nc.gz files have been extracted, cleaned up, and moved to the netcdf directory.")